# SENTINEL · Notebook 5c · Free public anchor on Bitcoin (OpenTimestamps)

The Polygon test faucets now ask for real ETH on Ethereum mainnet before they give free test tokens.
This notebook publishes your ledger's **Merkle roots** on a public blockchain **without any wallet, tokens or money**, using **OpenTimestamps**: a free, open service that anchors fingerprints (SHA-256 hashes) into the **Bitcoin** blockchain.

- Your roots go into a small text file, and its SHA-256 fingerprint is sent to public OpenTimestamps servers.
- Within a few hours, that fingerprint is included in a **Bitcoin block**. After that, nobody, not even you, can change the roots without it showing.
- Anyone can verify it at **opentimestamps.org**.

Your Polygon code from Notebook 5b stays ready: if you get test POL later, Notebook 5b still works.

Tap **☰ → Runtime → Run all** and allow Google Drive access.

## Step 1 · Load the tools and your ledger

In [1]:
!pip install -q opentimestamps-client
import json, os, sys, re, subprocess, importlib, hashlib
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/SENTINEL"
DATA_DIR = f"{BASE}/data"
sys.path.insert(0, f"{BASE}/code/ledger")
import ledger_model as lm
importlib.reload(lm)

if not os.path.exists(f"{DATA_DIR}/ledger_export.json"):
    print("❌ ledger_export.json not found → run Notebook 5 completely first.")
else:
    export = json.load(open(f"{DATA_DIR}/ledger_export.json"))
    anchors = json.load(open(f"{DATA_DIR}/anchors.json"))
    print(f"✅ Step 1 done: {len(export['records'])} ledger records, {len(export['batches'])} Merkle batches")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 13.4 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

## Step 2 · Recompute the roots from the ledger and write them to a file
We don't trust the stored roots: we **recompute** each batch's Merkle root from the records themselves, then write them into `merkle_roots.txt`.

In [ ]:
hashes = {r["seq"]: r["hash"] for r in export["records"]}
roots = [lm.merkle_root([hashes[s] for s in range(b["first_seq"], b["last_seq"] + 1)]) for b in export["batches"]]
stored = [b["merkle_root"] for b in export["batches"]]
print("Recomputed roots match the stored ones:", "✅ yes" if roots == stored else "❌ NO")

ROOTS_FILE = f"{DATA_DIR}/merkle_roots.txt"
OTS_FILE = ROOTS_FILE + ".ots"
content = "SENTINEL ledger Merkle roots (one per batch)\n" + "\n".join(
    f"batch {b['batch']} records {b['first_seq']}-{b['last_seq']} {root}" for b, root in zip(export["batches"], roots)) + "\n"
old = open(ROOTS_FILE).read() if os.path.exists(ROOTS_FILE) else None
if old != content:
    with open(ROOTS_FILE, "w") as f:
        f.write(content)
    if os.path.exists(OTS_FILE):
        os.remove(OTS_FILE)          # roots changed, so the old proof no longer applies
fingerprint = hashlib.sha256(content.encode()).hexdigest()
print(f"File fingerprint (SHA-256): {fingerprint}")
print("✅ Step 2 done: merkle_roots.txt saved in Drive → SENTINEL → data")

## Step 3 · Send the fingerprint to the public OpenTimestamps servers
This creates `merkle_roots.txt.ots`, the **proof file**. At first it says *pending*: the servers have accepted it and will put it into Bitcoin, usually within a few hours.

In [ ]:
def ots(*args):
    r = subprocess.run(["ots", *args], capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

if os.path.exists(OTS_FILE):
    print("Proof file already exists, not stamping again.")
else:
    print(ots("stamp", ROOTS_FILE))
if os.path.exists(OTS_FILE):
    info = ots("info", OTS_FILE)
    servers = sorted(set(re.findall(r"https://[\w\.\-/]+", info)))
    print(f"\nPending at {len(servers)} public servers:")
    for s in servers:
        print("  ", s)
    anchors.update({"method": "OpenTimestamps (Bitcoin)", "ots_file": "merkle_roots.txt.ots",
                    "fingerprint": fingerprint, "status": "pending Bitcoin confirmation"})
    json.dump(anchors, open(f"{DATA_DIR}/anchors.json", "w"), indent=1)
    export["anchor"] = anchors
    json.dump(export, open(f"{DATA_DIR}/ledger_export.json", "w"), indent=1)
    print("✅ Step 3 done: fingerprint submitted. Come back in a few hours for Step 4.")
else:
    print("❌ Stamping failed. Wait 5 minutes and run Step 3 again.")

## Step 4 · Upgrade the proof to a Bitcoin block *(run this again a few hours later)*
Bitcoin makes a block about every 10 minutes, and the servers batch fingerprints, so this usually completes within **1–12 hours**.
Run **Step 4 only** later (tap ▶ on it; if you get `NameError`, use **Run all**). When it's complete it shows a **Bitcoin block number**.

In [ ]:
if os.path.exists(OTS_FILE):
    print(ots("upgrade", OTS_FILE)[-300:])
    info = ots("info", OTS_FILE)
    blocks = sorted(set(int(b) for b in re.findall(r"BitcoinBlockHeaderAttestation\((\d+)\)", info)))
    if blocks:
        anchors.update({"status": "confirmed in Bitcoin", "bitcoin_block": blocks[0]})
        json.dump(anchors, open(f"{DATA_DIR}/anchors.json", "w"), indent=1)
        export["anchor"] = anchors
        json.dump(export, open(f"{DATA_DIR}/ledger_export.json", "w"), indent=1)
        print(f"\n✅ CONFIRMED: your Merkle roots are anchored in Bitcoin block #{blocks[0]}")
        print(f"   Block on a public explorer: https://mempool.space/block/{blocks[0]}")
        print("🎉 Notebook 5c complete. Send Claude a screenshot of this step.")
    else:
        print("\n⏳ Still pending: Bitcoin hasn't included it yet. Run Step 4 again in an hour or two.")
else:
    print("❌ No proof file yet: run Step 3 first.")

## Step 5 · How anyone can verify it (judges, auditors, donors)
1. In the **Google Drive** app open **SENTINEL → data** and download **merkle_roots.txt** and **merkle_roots.txt.ots**.
2. Open **opentimestamps.org** in Chrome.
3. In the **Stamp & Verify** box, add the **.ots** file, then the **.txt** file.
4. The site shows: *"Bitcoin block #… attests existence as of [date]"*.

If even one character of the roots file is changed, verification fails, and the roots come from every record in the ledger, so changing any record also breaks it.

## Say this to a judge
> *"Our ledger seals a Merkle root every 15 minutes. In the prototype we anchor the roots in the Bitcoin blockchain through OpenTimestamps, which is free and needs no wallet, and our code also supports anchoring on Polygon Amoy. Here is the proof: Bitcoin block number …. If anyone changes a single record, the recomputed roots no longer match what Bitcoin recorded."*